In [ ]:
🔧 COMMON HELPERS (USE EVERYWHERE)
🧩 8-PUZZLE NEIGHBORS
def get_neighbors(state):
    neighbors = []
    idx = state.index(0)
    x, y = idx // 3, idx % 3

    moves = [(-1,0),(1,0),(0,-1),(0,1)]

    for dx, dy in moves:
        nx, ny = x + dx, y + dy
        if 0 <= nx < 3 and 0 <= ny < 3:
            new_idx = nx*3 + ny
            new_state = list(state)
            new_state[idx], new_state[new_idx] = new_state[new_idx], new_state[idx]
            neighbors.append(tuple(new_state))
    return neighbors


neighbors
📏 MANHATTAN HEURISTIC
def manhattan(state, goal):
    dist = 0
    for i in range(9):
        if state[i] != 0:
            goal_idx = goal.index(state[i])
            x1, y1 = i//3, i%3
            x2, y2 = goal_idx//3, goal_idx%3
            dist += abs(x1-x2) + abs(y1-y2)
    return dist

🟦 1. BFS
from collections import deque

def bfs(start, goal):
    queue = deque([(start, [])])
    visited = set()

    while queue:
        state, path = queue.popleft()

        if state == goal:
            return path

        visited.add(state)

        for neighbor in get_neighbors(state):
            if neighbor not in visited:
                queue.append((neighbor, path + [neighbor]))

🟨 2. DFS
def dfs(start, goal):
    stack = [(start, [])]
    visited = set()

    while stack:
        state, path = stack.pop()

        if state == goal:
            return path

        if state not in visited:
            visited.add(state)

            for neighbor in get_neighbors(state):
                stack.append((neighbor, path + [neighbor]))


🟩 3. UCS
import heapq

def ucs(start, goal):
    pq = [(0, start, [])]
    visited = set()

    while pq:
        cost, state, path = heapq.heappop(pq)

        if state == goal:
            return path

        if state not in visited:
            visited.add(state)

            for neighbor in get_neighbors(state):
                heapq.heappush(pq, (cost+1, neighbor, path+[neighbor]))

🟧 4. GREEDY BEST FIRST
def greedy(start, goal):
    pq = [(manhattan(start, goal), start, [])]
    visited = set()

    while pq:
        _, state, path = heapq.heappop(pq)

        if state == goal:
            return path

        visited.add(state)

        for neighbor in get_neighbors(state):
            if neighbor not in visited:
                h = manhattan(neighbor, goal)
                heapq.heappush(pq, (h, neighbor, path+[neighbor]))

🟩 5. A*
def a_star(start, goal):
    pq = [(0, start, [])]
    g_cost = {start: 0}

    while pq:
        _, state, path = heapq.heappop(pq)

        if state == goal:
            return path

        for neighbor in get_neighbors(state):
            temp_g = g_cost[state] + 1

            if neighbor not in g_cost or temp_g < g_cost[neighbor]:
                g_cost[neighbor] = temp_g
                f = temp_g + manhattan(neighbor, goal)
                heapq.heappush(pq, (f, neighbor, path+[neighbor]))

🟪 6. RBFS
def rbfs(start, goal):

    def helper(state, g, f_limit, path, visited):
        if state == goal:
            return path, g

        successors = []

        for neighbor in get_neighbors(state):
            if neighbor not in visited:
                new_g = g + 1
                f = new_g + manhattan(neighbor, goal)
                successors.append((f, neighbor, new_g))

        if not successors:
            return None, float('inf')

        while True:
            successors.sort(key=lambda x: x[0])
            best_f, best_state, best_g = successors[0]

            if best_f > f_limit:
                return None, best_f

            alt = successors[1][0] if len(successors) > 1 else float('inf')

            visited.add(best_state)

            result, best_f = helper(
                best_state,
                best_g,
                min(f_limit, alt),
                path + [best_state],
                visited
            )

            if result is not None:
                return result, best_f

🟥 8. IDA* (IMPORTANT)

def ida_star(start, goal):

    def search(state, g, threshold, path):
        f = g + manhattan(state, goal)

        if f > threshold:
            return f

        if state == goal:
            return path

        min_threshold = float('inf')

        for neighbor in get_neighbors(state):
            if neighbor not in path:
                temp = search(neighbor, g+1, threshold, path+[neighbor])

                if isinstance(temp, list):
                    return temp

                min_threshold = min(min_threshold, temp)

        return min_threshold

    threshold = manhattan(start, goal)

    while True:
        temp = search(start, 0, threshold, [start])

        if isinstance(temp, list):
            return temp

        if temp == float('inf'):
            return None

        threshold = temp